# 05 3D Dual-Stream PET/CT Alignment Train

Inputs: output dataset from notebook `01_build_25d_cache.ipynb` with
`BUILD_3D_PATCH_CACHE=True`.

This notebook trains a compact PET encoder + CT encoder + fusion model
against report embeddings. It is the practical 3D alignment stage for T4.
Gate tối thiểu: val loss decreases, no OOM, 3D retrieval/clinical metrics
improve over the 2.5D retrieval baseline in notebook 02.

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings("ignore")

SMOKE_MODE = True
EPOCHS = 1 if SMOKE_MODE else 5
BATCH_SIZE = 1
GRAD_ACCUM = 8 if SMOKE_MODE else 16
OUTPUT_ROOT = Path("/kaggle/working/vimedpet_q1_outputs") if Path("/kaggle/working").exists() else Path("vimedpet_q1_outputs")
OUT_DIR = OUTPUT_ROOT / "05_3d_dual_stream"
OUT_DIR.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def locate_cache_meta():
    roots = list(Path("/kaggle/input").rglob("q1_cache_meta.json")) if Path("/kaggle/input").exists() else []
    roots += list(Path.cwd().rglob("q1_cache_meta.json"))
    if not roots: raise FileNotFoundError("q1_cache_meta.json not found")
    return sorted(roots, key=lambda p: len(str(p)))[0]
meta_path = locate_cache_meta()
CACHE_DIR = meta_path.parent
meta = json.loads(meta_path.read_text())
if not meta.get("patch_mmap_path"):
    raise FileNotFoundError("3D patch cache missing. Re-run notebook 01 with BUILD_3D_PATCH_CACHE=True.")
clean = pd.read_csv(CACHE_DIR / "q1_clean_manifest_for_cache.csv")
cache_index = pd.read_csv(CACHE_DIR / "q1_cache_index.csv")
clean = clean.merge(cache_index[cache_index.cache_ok][["case_id", "row_index"]], on="case_id", how="inner")
patch_name = meta["patch_mmap_path"]
m = __import__("re").search(r"d(\d+)_s(\d+)", patch_name)
D, S = int(m.group(1)), int(m.group(2))
vol = np.memmap(CACHE_DIR / patch_name, dtype=np.uint8, mode="r", shape=(meta["n"], 2, D, S, S))
print("3D cache:", CACHE_DIR / patch_name, vol.shape)

In [ ]:
texts = clean["target_text"].fillna(clean["report_text_clean"]).astype(str).tolist()
tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1,2), min_df=2)
X = tfidf.fit_transform(texts)
svd = TruncatedSVD(n_components=min(256, X.shape[1]-1, len(clean)-1), random_state=391)
text_emb = torch.tensor(normalize(svd.fit_transform(X)), dtype=torch.float32)

class VolDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        x = torch.tensor(vol[int(row.row_index)].astype(np.float32) / 255.0)
        y = text_emb[clean.index[clean.case_id == row.case_id][0]]
        return x, y, row.case_id

class Branch(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(1, 8, 3, stride=2, padding=1), nn.BatchNorm3d(8), nn.ReLU(),
            nn.Conv3d(8, 16, 3, stride=2, padding=1), nn.BatchNorm3d(16), nn.ReLU(),
            nn.Conv3d(16, 32, 3, stride=2, padding=1), nn.BatchNorm3d(32), nn.ReLU(),
            nn.AdaptiveAvgPool3d(1),
        )
    def forward(self, x): return self.net(x).flatten(1)

class DualStream(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.ct = Branch(); self.pet = Branch()
        self.fuse = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, out_dim))
    def forward(self, x):
        ct = self.ct(x[:,0:1]); pet = self.pet(x[:,1:2])
        return F.normalize(self.fuse(torch.cat([ct, pet], dim=1)), dim=1)

def contrastive_loss(img, txt, temp=0.07):
    logits = img @ txt.T / temp
    labels = torch.arange(len(img), device=img.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

train_df = clean[clean.clean_split == "train"].copy()
val_df = clean[clean.clean_split == "validation"].copy()
if SMOKE_MODE:
    train_df = train_df.sample(min(32, len(train_df)), random_state=391)
    val_df = val_df.sample(min(32, len(val_df)), random_state=392)
train_loader = DataLoader(VolDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(VolDataset(val_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = DualStream(text_emb.shape[1]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
best_val = 1e9
for epoch in range(EPOCHS):
    model.train(); losses = []
    opt.zero_grad()
    for step, (x, y, _) in enumerate(train_loader, 1):
        x, y = x.to(device), y.to(device)
        loss = contrastive_loss(model(x), F.normalize(y, dim=1)) / GRAD_ACCUM
        loss.backward()
        if step % GRAD_ACCUM == 0:
            opt.step(); opt.zero_grad()
        losses.append(float(loss.item() * GRAD_ACCUM))
    model.eval(); vloss = []
    with torch.no_grad():
        for x, y, _ in val_loader:
            x, y = x.to(device), y.to(device)
            vloss.append(float(contrastive_loss(model(x), F.normalize(y, dim=1)).item()))
    avg_val = float(np.mean(vloss))
    print({"epoch": epoch, "train_loss": float(np.mean(losses)), "val_loss": avg_val})
    if avg_val < best_val:
        best_val = avg_val
        torch.save(model.state_dict(), OUT_DIR / "dual_stream_best.pt")
(OUT_DIR / "dual_stream_metrics.json").write_text(json.dumps({"best_val_loss": best_val, "smoke_mode": SMOKE_MODE}, indent=2), encoding="utf-8")
print("Saved:", OUT_DIR)